In [1]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
import os
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer
from sklearn.preprocessing import MinMaxScaler
from utils.paths import DATASETS_RAW_DIR, DATASETS_CLEANED_DIR

c:\Users\Wiktor\Desktop\Wiktor\pliki\projekt_naukowo_wdrożeniowy\projekt-naukowo-wdrozeniowy\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
path = kagglehub.dataset_download("mubashirrahim/wind-power-generation-data-forecasting")
datasets = []

os.makedirs(DATASETS_RAW_DIR, exist_ok=True)

dataset_files = []
for file in os.listdir(path):
    if file.endswith(".csv"): dataset_files.append(file)

for file in dataset_files:
    source_path = os.path.join(path, file)
    destination_path = os.path.join(DATASETS_RAW_DIR, file)
    df = pd.read_csv(source_path)
    df.columns = ['time', 'temperature_2m', 'relative_humidity_2m', 'dew_point_2m', 'wind_speed_10m', 'wind_speed_100m', 
                    'wind_direction_10m', 'wind_direction_100m', 'wind_gusts_10m', 'power']
    df.to_csv(destination_path, index=False)
    datasets.append(df)

100%|██████████| 3.32M/3.32M [00:00<00:00, 3.62MB/s]

Extracting files...


In [3]:
for i, df in enumerate(datasets):
    print(f"Location {i+1}: {df.shape[0]} rows, {df.shape[1]} columns")
    print(f"{df.info()}\n")
    print(f"{df.isna().sum()}\n")
    print(f"{df.describe()}\n")
    print(f"#" * 40 + "\n")

Location 1: 43800 rows, 10 columns
<class 'pandas.DataFrame'>
RangeIndex: 43800 entries, 0 to 43799
Data columns (total 10 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   time                  43800 non-null  str    
 1   temperature_2m        43800 non-null  float64
 2   relative_humidity_2m  43800 non-null  int64  
 3   dew_point_2m          43800 non-null  float64
 4   wind_speed_10m        43800 non-null  float64
 5   wind_speed_100m       43800 non-null  float64
 6   wind_direction_10m    43800 non-null  int64  
 7   wind_direction_100m   43800 non-null  int64  
 8   wind_gusts_10m        43800 non-null  float64
 9   power                 43800 non-null  float64
dtypes: float64(6), int64(3), str(1)
memory usage: 3.3 MB
None

time                    0
temperature_2m          0
relative_humidity_2m    0
dew_point_2m            0
wind_speed_10m          0
wind_speed_100m         0
wind_direction_10m      0
wind_dire

In [4]:
# checking for time gaps and duplicates in the data
for i, df in enumerate(datasets):
    df['time'] = pd.to_datetime(df['time'], format='%Y-%m-%d %H:%M:%S')
    time_diffs = df['time'].diff()
    gaps = time_diffs[time_diffs != pd.Timedelta(hours=1)]
    gaps.drop(0, inplace=True)  # first row will be NaT - no previous timestamp to compare with

    if gaps.empty: print(f"No time gaps found for location {i+1}")
    else: print(f"Time gaps found for location {i+1}")

    if df['time'].is_unique: print(f"All timestamps are unique for location {i+1}")
    else: print(f"Duplicate timestamps for location {i+1}")

    print(f"\n{'#' * 40}\n")

No time gaps found for location 1
All timestamps are unique for location 1

########################################

No time gaps found for location 2
All timestamps are unique for location 2

########################################

No time gaps found for location 3
All timestamps are unique for location 3

########################################

No time gaps found for location 4
All timestamps are unique for location 4

########################################



In [5]:
# checking for potentially incorrect power values (e.g. failures or interruptions in operation)

assumed_v_cutin = 3 # typical cut_in speed form wind turbines is around 3 m/s

for i, df in enumerate(datasets):
    print(f"Location {i+1}:")
    print(f"{df[df['power'] <= 0].shape[0]} rows with power <= 0")
    print(f"{df[df['power'] > 1].shape[0]} rows with power > 1")

    print(f"{df[(df['power'] <= 0) & (df['wind_speed_100m'] > assumed_v_cutin)].shape[0]} rows with power <= 0 and wind speed > {assumed_v_cutin} m/s")

    print(f"\n{'#' * 40}\n")

Location 1:
101 rows with power <= 0
0 rows with power > 1
94 rows with power <= 0 and wind speed > 3 m/s

########################################

Location 2:
1113 rows with power <= 0
0 rows with power > 1
903 rows with power <= 0 and wind speed > 3 m/s

########################################

Location 3:
2989 rows with power <= 0
0 rows with power > 1
2359 rows with power <= 0 and wind speed > 3 m/s

########################################

Location 4:
1465 rows with power <= 0
0 rows with power > 1
548 rows with power <= 0 and wind speed > 3 m/s

########################################



In [6]:
features_to_impute = ['temperature_2m', 'relative_humidity_2m', 'wind_speed_100m', 'wind_direction_100m', 'power']
nan_idx_list = []

for i, df in enumerate(datasets):
    df.loc[(df['power'] <= 0) & (df['wind_speed_100m'] > assumed_v_cutin), 'power'] = np.nan
    nan_idx = df[df['power'].isna()].index
    nan_idx_list.append(nan_idx)
    
    if len(nan_idx) > 0:
        scaler = MinMaxScaler()
        imputer = KNNImputer(n_neighbors=5, weights='distance')

        df_scaled = scaler.fit_transform(df[features_to_impute])
        df_imputed = imputer.fit_transform(df_scaled)
        df_imputed = scaler.inverse_transform(df_imputed)
        df_imputed_final = pd.DataFrame(df_imputed, columns=features_to_impute, index=df.index)

        df.loc[nan_idx, 'power'] = df_imputed_final.loc[nan_idx, 'power']

    datasets[i] = df



In [7]:
for i, df in enumerate(datasets):
    print(f"Location {i+1}:")
    print(f"{df[df['power'] <= 0].shape[0]} rows with power <= 0")
    print(f"{df[df['power'] > 1].shape[0]} rows with power > 1")
    print(f"{df[(df['power'] <= 0) & (df['wind_speed_100m'] > assumed_v_cutin)].shape[0]} rows with power <= 0 and wind speed > {assumed_v_cutin} m/s")

    repaired = df.loc[(df['wind_speed_100m'] > assumed_v_cutin) & (df.index.isin(nan_idx_list[i])), 'power']
    print(repaired.describe())
    
    print(f"\n{'#' * 40}\n")

Location 1:
7 rows with power <= 0
0 rows with power > 1
0 rows with power <= 0 and wind speed > 3 m/s
count    94.000000
mean      0.432526
std       0.176517
min       0.111620
25%       0.293422
50%       0.406668
75%       0.528175
max       0.876062
Name: power, dtype: float64

########################################

Location 2:
210 rows with power <= 0
0 rows with power > 1
0 rows with power <= 0 and wind speed > 3 m/s
count    903.000000
mean       0.223598
std        0.167464
min        0.006339
25%        0.093764
50%        0.180474
75%        0.313141
max        0.792828
Name: power, dtype: float64

########################################

Location 3:
631 rows with power <= 0
0 rows with power > 1
1 rows with power <= 0 and wind speed > 3 m/s
count    2359.000000
mean        0.355671
std         0.228034
min         0.000000
25%         0.159112
50%         0.327120
75%         0.519838
max         0.962696
Name: power, dtype: float64

####################################

In [8]:
# correcting records with power > 0 but wind speed above cut_in speed
for i, df in enumerate(datasets):
    invalid_records = (df['wind_speed_100m'] < assumed_v_cutin) & (df['power'] > 0)
    
    df.loc[invalid_records, 'power'] = 0
    datasets[i] = df

In [9]:
os.makedirs(DATASETS_CLEANED_DIR, exist_ok=True)

for i, df in enumerate(datasets):
    df.to_csv(os.path.join(DATASETS_CLEANED_DIR, f"Location{i+1}.csv"), index=False)